In [ ]:
%pip install --quiet beautifulsoup4 requests pandas selenium webdriver-manager

In [ ]:
## NEED TO SPECIFY WHICH VERSION OF PYTHON TO USE 3.15.1

In [ ]:
from urllib.parse import urlparse
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
import json
import logging
import requests
from urllib.robotparser import RobotFileParser

In [ ]:
BASE_URL = "https://forums.moneysavingexpert.com/"
FORUM_URL = "https://forums.moneysavingexpert.com/categories/benefits-tax-credits"
NUMBER_OF_THREADS_TO_SCRAPE = 30

In [ ]:
# First we should create a ethical scraper which respects the robots.txt file and the terms of service of the website.

class RespectfulScraper:
    """
    A web scraper that respects robots.txt rules and rate limits.

    This class automatically:
    - Parses robots.txt for the target domain
    - Enforces crawl delays between requests
    - Identifies itself with a descriptive User-Agent string
    - Logs all activity for transparency and debugging
    """

    def __init__(self, base_url: str, default_delay: int = 3):
        self.base_url = base_url.rstrip("/")

        # Create a session for connection reuse and consistent headers
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "BPP-DataScience-Bot/1.0 (educational project)"
        })

        # Set up logging
        self.logger = logging.getLogger(f"scraper.{urlparse(base_url).hostname}")
        self.logger.setLevel(logging.INFO)

          # Parse robots.txt (fetched with our own session/User-Agent)
        robots_url = f"{self.base_url}/robots.txt"
        self.rp = RobotFileParser()
        self.rp.set_url(robots_url)

        resp = self.session.get(robots_url, timeout=10)
        
        if resp.status_code == 200:
            self.rp.parse(resp.text.splitlines())
            self.logger.info(f"Response {resp.status_code}: Successfully parsed robots.txt")
        elif resp.status_code in (401, 403):
            self.rp.disallow_all = True
            self.logger.warning(f"robots.txt returned {resp.status_code}; treating site as off-limits")
        elif 400 <= resp.status_code < 500:
            self.rp.allow_all = True
            self.logger.info(f"No robots.txt ({resp.status_code}); everything allowed")
        else:
            raise RuntimeError(f"Couldn't fetch robots.txt ({resp.status_code}); not crawling")

        delay = self.rp.crawl_delay(self.session.headers["User-Agent"])
        self.crawl_delay = float(delay) if delay else default_delay

    def can_fetch(self, url: str) -> bool:
        """Check if a URL is allowed by robots.txt."""
        return self.rp.can_fetch(self.session.headers["User-Agent"], url)

    def get(self, url: str) -> Optional[BeautifulSoup]:
        """
        Fetch and parse a page, respecting robots.txt and crawl delays.

        Returns a BeautifulSoup object if successful, None otherwise.
        """
        if not self.can_fetch(url):
            self.logger.warning(f"Blocked by robots.txt: {url}")
            return None

        try:
            self.logger.info(f"Fetching {url}")
            response = self.session.get(url, timeout=10)
            response.raise_for_status()

            # Respect crawl delay
            time.sleep(self.crawl_delay)

            return BeautifulSoup(response.text, "html.parser")

        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error fetching {url}: {e}")
            return None

In [ ]:
mse_scraper = RespectfulScraper(base_url=BASE_URL, default_delay=3)

In [ ]:
# 1. Create scraper and check the robots.txt file etc
# 2. Scrpae the url provided and get the thread data (url, pinned, announcement, countViews, countComments)
    ## Count the number of records if less than the number specified in the config, then scrape the next page until the number of records is reached or there are no more pages to scrape.
    
def scrape_forum(scraper, forum_url: str, max_threads: int = 10) -> pd.DataFrame:
    count_of_threads = 0
    
    # Extract the number of posts from the current page
    thread_data = []
    
    url_list = [forum_url]
    
    for url in url_list:
        if count_of_threads >= max_threads:
            break
        else:
            soup = scraper.get(url)
            for dataprops in soup.find_all("div", attrs={"data-props": True}):
                props = json.loads(dataprops["data-props"])

                for thread in props.get("discussions", []):
                    if count_of_threads >= max_threads:
                        break
                    else:
                        if thread.get("pinned") is not True and thread.get("announce") is not True:
                            thread_data.append({
                                "url": thread.get("url"),
                                "pinned": thread.get("pinned"),
                                "announcement": thread.get("announce"),
                                "count_views": thread.get("countViews"),
                                "count_comments": thread.get("countComments"),
                            })
                            count_of_threads += 1
            if count_of_threads >= max_threads:
                break
            else: 
                next_page_link = soup.find("div", class_="PageControls Top").find("a", class_="Next")["href"]
                url_list.append(next_page_link)
                
    return pd.DataFrame(thread_data)
    

In [ ]:
threads_df = scrape_forum(mse_scraper, FORUM_URL, max_threads=NUMBER_OF_THREADS_TO_SCRAPE)

In [ ]:
threads_df

In [ ]:
def scrape_threads(scraper, threads_df: pd.DataFrame) -> pd.DataFrame:
    
    threads_df = threads_df.copy()
    
    for url in threads_df["url"]:
        soup = scraper.get(url)
        # Get the first post content from the thread page
        first_post_content = soup.find("div", class_="Message userContent")
        threads_df.loc[threads_df["url"] == url, "first_post_content"] = str(first_post_content)
    return threads_df

In [ ]:
posts = scrape_threads(mse_scraper, threads_df)


In [ ]:
posts